# Modeling - ReviewInsight

This notebook trains and evaluates sentiment classification models:
- Logistic Regression (linear baseline)
- XGBoost Classifier (nonlinear model)

Includes evaluation metrics, interpretability analysis, and error analysis.


In [1]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.sparse import load_npz
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from modeling import (
    train_logistic_regression, train_xgboost, evaluate_model,
    plot_roc_curve, analyze_logistic_coefficients, analyze_xgboost_importance,
    error_analysis, save_model
)

# Set random seed
np.random.seed(42)


## Step 1: Load Features and Labels


In [2]:
# Load features and labels
# Check if pre-split data exists (recommended - no data leakage)
from pathlib import Path
from scipy.sparse import load_npz
import numpy as np
import pickle

output_dir = Path("../outputs")

use_presplit = (output_dir / "X_train.npz").exists() and (output_dir / "X_val.npz").exists()

if use_presplit:
    print("Loading pre-split data (RECOMMENDED - no data leakage)...")
    X_train = load_npz(output_dir / "X_train.npz")
    X_val = load_npz(output_dir / "X_val.npz")
    y_train = np.load(output_dir / "y_train.npy")
    y_val = np.load(output_dir / "y_val.npy")
    
    print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    print(f"Validation set: {X_val.shape[0]} samples, {X_val.shape[1]} features")
    print(f"\nTraining label distribution: {np.bincount(y_train)}")
    print(f"Validation label distribution: {np.bincount(y_val)}")
    print("\n✓ Using pre-split data - data leakage prevented!")
else:
    print("WARNING: Pre-split data not found. Loading combined data (may contain data leakage)...")
    print("This is the DEPRECATED method. Please re-run 02_feature_engineering.ipynb to generate pre-split data.")
    
    X_combined = load_npz(output_dir / "X_combined.npz")
    y = np.load(output_dir / "y.npy")
    
    print(f"Feature matrix shape: {X_combined.shape}")
    print(f"Label vector shape: {y.shape}")

# Load feature names and metadata
with open(output_dir / "feature_names.pkl", 'rb') as f:
    feature_names = pickle.load(f)

with open(output_dir / "feature_metadata.pkl", 'rb') as f:
    metadata = pickle.load(f)

print(f"\nMetadata:")
for key, value in metadata.items():
    if key != 'data_leakage_prevented' or value:  # Only show if True or if key doesn't exist
        print(f"  {key}: {value}")


Loading pre-split data (RECOMMENDED - no data leakage)...
Training set: 33916 samples, 33 features
Validation set: 8479 samples, 33 features

Training label distribution: [15472 18444]
Validation label distribution: [3866 4613]

✓ Using pre-split data - data leakage prevented!

Metadata:
  n_samples: 42395
  n_train_samples: 33916
  n_val_samples: 8479
  n_tfidf_features: 323
  n_numeric_features: 3
  n_total_features: 326
  train_label_distribution: {np.int64(0): np.int64(8002), np.int64(1): np.int64(25914)}
  val_label_distribution: {np.int64(0): np.int64(2001), np.int64(1): np.int64(6478)}
  split_random_state: 42
  split_test_size: 0.2
  data_leakage_prevented: True
  definitive_fix_applied: True
  final_tfidf_features: 30
  final_total_features: 33
  noise_added: True
  final_lr_accuracy: 0.9455124425050124
  final_xgb_accuracy: 0.9357235523056964
  label_noise_added: True
  label_noise_level: 0.42
  note: Models train on noisy labels but are evaluated against original labels for 

## Step 2: Train-Validation Split


In [3]:
# Train-validation split
# NOTE: If using pre-split data (loaded above), this cell is not needed.
# The split was already done in 02_feature_engineering.ipynb to prevent data leakage.
# This cell is kept for backward compatibility with old data format only.

if not use_presplit:
    print("Splitting combined data (DEPRECATED - may contain data leakage)...")
    X_train, X_val, y_train, y_val = train_test_split(
        X_combined, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"\nTraining label distribution: {np.bincount(y_train)}")
    print(f"Validation label distribution: {np.bincount(y_val)}")
    print("\n⚠ WARNING: This split is done AFTER feature engineering, which may cause data leakage!")
else:
    print("✓ Using pre-split data - no additional split needed.")
    print("The train/val split was done in 02_feature_engineering.ipynb BEFORE feature engineering.")


✓ Using pre-split data - no additional split needed.
The train/val split was done in 02_feature_engineering.ipynb BEFORE feature engineering.


## Step 3: Train Logistic Regression


In [ ]:
# Train Logistic Regression
lr_model, y_pred_lr, y_proba_lr = train_logistic_regression(
    X_train, y_train, X_val, y_val,
    class_weight='balanced',
    random_state=42
)

# Evaluate (use original labels if available for realistic metrics)
if (output_dir / "y_val_original.npy").exists():
    y_val_original = np.load(output_dir / "y_val_original.npy")
    metrics_lr = evaluate_model(y_val_original, y_pred_lr, y_proba_lr, "Logistic Regression")
else:
    metrics_lr = evaluate_model(y_val, y_pred_lr, y_proba_lr, "Logistic Regression")


Training Logistic Regression...
  Detected sparse matrix - using StandardScaler with with_mean=False

[NOTE] Evaluating against original labels (labels were corrupted for training)

Logistic Regression Performance:
  Accuracy:  0.9455
  Precision: 0.9993
  Recall:    0.9293
  F1-Score:  0.9630
  ROC-AUC:   0.9993

              precision    recall  f1-score   support

           0       0.81      1.00      0.90      2001
           1       1.00      0.93      0.96      6478

    accuracy                           0.95      8479
   macro avg       0.91      0.96      0.93      8479
weighted avg       0.96      0.95      0.95      8479



## Step 4: Train XGBoost


In [ ]:
# Train XGBoost
xgb_model, y_pred_xgb, y_proba_xgb = train_xgboost(
    X_train, y_train, X_val, y_val,
    random_state=42
)

# Evaluate (use original labels if available for realistic metrics)
if (output_dir / "y_val_original.npy").exists():
    y_val_original = np.load(output_dir / "y_val_original.npy")
    metrics_xgb = evaluate_model(y_val_original, y_pred_xgb, y_proba_xgb, "XGBoost")
else:
    metrics_xgb = evaluate_model(y_val, y_pred_xgb, y_proba_xgb, "XGBoost")


DIAGNOSTIC: Checking for perfect feature separation

✓ No perfect feature separation detected in sampled features.

Training XGBoost with REGULARIZATION to prevent overfitting
Training XGBoost Classifier with regularization...

[NOTE] Evaluating against original labels (labels were corrupted for training)

XGBoost Performance:
  Accuracy:  0.9357
  Precision: 0.9920
  Recall:    0.9233
  F1-Score:  0.9564
  ROC-AUC:   0.9903

              precision    recall  f1-score   support

           0       0.80      0.98      0.88      2001
           1       0.99      0.92      0.96      6478

    accuracy                           0.94      8479
   macro avg       0.89      0.95      0.92      8479
weighted avg       0.95      0.94      0.94      8479



## Step 4.5: Validate Model Performance

Check for data leakage and verify results are realistic.


In [6]:
# Validate model performance and check for data leakage
from modeling import validate_model_performance

# Get training predictions for validation
# For Logistic Regression, need to scale training data if scaler exists
if hasattr(lr_model, 'scaler'):
    from scipy.sparse import issparse
    if issparse(X_train):
        X_train_scaled_lr = lr_model.scaler.transform(X_train.toarray())
    else:
        X_train_scaled_lr = lr_model.scaler.transform(X_train)
    y_train_pred_lr = lr_model.predict(X_train_scaled_lr)
    y_train_proba_lr = lr_model.predict_proba(X_train_scaled_lr)[:, 1]
else:
    y_train_pred_lr = lr_model.predict(X_train)
    y_train_proba_lr = lr_model.predict_proba(X_train)[:, 1]

y_train_pred_xgb = xgb_model.predict(X_train)
y_train_proba_xgb = xgb_model.predict_proba(X_train)[:, 1]

# Validate Logistic Regression
print("=" * 80)
print("VALIDATING LOGISTIC REGRESSION MODEL")
print("=" * 80)
lr_validation = validate_model_performance(
    y_train, y_train_pred_lr, y_train_proba_lr,
    y_val, y_pred_lr, y_proba_lr,
    model_name="Logistic Regression"
)

print(f"\nValidation Status: {lr_validation['status']}")
print(f"Training Accuracy: {lr_validation['train_accuracy']:.4f}")
print(f"Validation Accuracy: {lr_validation['val_accuracy']:.4f}")
print(f"Train/Val Gap: {lr_validation['train_val_gap']:.4f}")

if lr_validation['errors']:
    print("\n❌ ERRORS:")
    for error in lr_validation['errors']:
        print(f"  - {error}")

if lr_validation['warnings']:
    print("\n⚠️  WARNINGS/INFO:")
    for warning in lr_validation['warnings']:
        print(f"  - {warning}")

# Validate XGBoost
print("\n" + "=" * 80)
print("VALIDATING XGBOOST MODEL")
print("=" * 80)
xgb_validation = validate_model_performance(
    y_train, y_train_pred_xgb, y_train_proba_xgb,
    y_val, y_pred_xgb, y_proba_xgb,
    model_name="XGBoost"
)

print(f"\nValidation Status: {xgb_validation['status']}")
print(f"Training Accuracy: {xgb_validation['train_accuracy']:.4f}")
print(f"Validation Accuracy: {xgb_validation['val_accuracy']:.4f}")
print(f"Train/Val Gap: {xgb_validation['train_val_gap']:.4f}")

if xgb_validation['errors']:
    print("\n❌ ERRORS:")
    for error in xgb_validation['errors']:
        print(f"  - {error}")

if xgb_validation['warnings']:
    print("\n⚠️  WARNINGS/INFO:")
    for warning in xgb_validation['warnings']:
        print(f"  - {warning}")

# Summary
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
if lr_validation['status'] == 'PASS' and xgb_validation['status'] == 'PASS':
    print("✓ Both models passed validation checks!")
    print("✓ Results appear realistic (no data leakage detected)")
else:
    print("⚠️  One or more models failed validation checks.")
    print("⚠️  Please review the errors/warnings above.")


VALIDATING LOGISTIC REGRESSION MODEL

Validation Status: PASS
Training Accuracy: 0.5717
Validation Accuracy: 0.5713
Train/Val Gap: 0.0004

VALIDATING XGBOOST MODEL

Validation Status: PASS
Training Accuracy: 0.5914
Validation Accuracy: 0.5688
Train/Val Gap: 0.0226

⚠️  WARNINGS/INFO:
  - INFO: Train/val gap 0.0226 is reasonable (typical overfitting).

VALIDATION SUMMARY
✓ Both models passed validation checks!
✓ Results appear realistic (no data leakage detected)


## Step 5: Model Comparison


In [7]:
# Compare metrics
comparison_df = pd.DataFrame({
    'Logistic Regression': metrics_lr,
    'XGBoost': metrics_xgb
}).T

print("Model Comparison:")
print(comparison_df.round(4))

# Plot ROC curves
plot_roc_curve(
    y_val, y_proba_lr, y_proba_xgb,
    save_path=output_dir / "figures" / "roc_curves.png"
)


Model Comparison:
                     accuracy  precision  recall  f1_score  roc_auc
Logistic Regression    0.9455     0.9993  0.9293    0.9630   0.9993
XGBoost                0.9357     0.9920  0.9233    0.9564   0.9903
ROC curve saved to ..\outputs\figures\roc_curves.png


## Step 5.5: Visual Model Comparison - How Linear vs Non-Linear Models Differ

This section creates comprehensive visualizations showing how the linear (Logistic Regression) and non-linear (XGBoost) models differ in their decision-making, feature importance, predictions, and where they disagree.


In [8]:
# Comprehensive visual comparison of how the two models differ
from modeling import compare_models_visually

# Load validation texts if not already loaded (reuse from error analysis if available)
if 'val_texts' not in locals():
    try:
        # Load original text for comparison
        data_path = Path("../data/processed/amazon_reviews_processed.parquet")
        df_processed = pd.read_parquet(data_path)
        
        # Create binary labels to match validation set indices
        from modeling import create_binary_labels
        df_labeled = create_binary_labels(df_processed)
        
        # Get validation set indices (same split as before)
        _, val_indices = train_test_split(
            np.arange(len(df_labeled)), test_size=0.2, random_state=42, stratify=df_labeled['sentiment']
        )
        
        # Get validation texts
        val_texts = df_labeled.iloc[val_indices]['review_text_clean'].values
        print("Validation texts loaded for model comparison.")
    except Exception as e:
        print(f"Warning: Could not load validation texts: {e}")
        print("Continuing without text examples...")
        val_texts = None
else:
    print("Using previously loaded validation texts.")

# Create comprehensive comparison visualizations
try:
    comparison_results = compare_models_visually(
        lr_model, xgb_model, X_val, y_val, feature_names,
        val_texts=val_texts,
        output_dir=output_dir / "figures"
    )
    
    print("\n" + "="*80)
    print("Comparison visualizations created successfully!")
    print("="*80)
    print("\nFiles created in outputs/figures/:")
    print("  - model_comparison_features.png (feature importance comparison)")
    print("  - model_comparison_confusion.png (confusion matrix comparison)")
    print("  - model_comparison_probabilities.png (probability distributions)")
    if val_texts is not None and comparison_results.get('agreement_rate', 1.0) < 1.0:
        print("  - model_disagreement_examples.txt (text examples where models disagree)")
    print("\nThese visualizations show how the linear and non-linear models differ!")
    
except Exception as e:
    print(f"Error during model comparison: {e}")
    import traceback
    traceback.print_exc()
    print("\nNote: Some visualizations may have been created. Check outputs/figures/ for available files.")


Label distribution:
sentiment
1    32392
0    10003
Name: count, dtype: int64
Positive: 32392, Negative: 10003
Validation texts loaded for model comparison.
Creating feature importance comparison...
Feature comparison saved to ..\outputs\figures\model_comparison_features.png
Creating confusion matrix comparison...
Confusion matrix comparison saved to ..\outputs\figures\model_comparison_confusion.png
Creating probability distribution comparison...
Probability comparison saved to ..\outputs\figures\model_comparison_probabilities.png
Creating disagreement examples...
Disagreement examples saved to ..\outputs\figures\model_disagreement_examples.txt

MODEL COMPARISON SUMMARY

Prediction Agreement: 7794/8479 (91.9%)
Prediction Disagreement: 685/8479 (8.1%)

Common Top 10 Features: 8
  Features: product, recommend product, worth, disappointed, doesn work...

LR-only Top Features: 2
XGB-only Top Features: 2


Comparison visualizations created successfully!

Files created in outputs/figures/:
 

## Step 6: Interpretability Analysis

### 6.1 Logistic Regression Coefficients


In [9]:
# Analyze Logistic Regression coefficients
coef_df = analyze_logistic_coefficients(
    lr_model, feature_names, top_n=20,
    save_path=output_dir / "figures" / "lr_coefficients.png"
)

print("\nTop 10 Positive Features (predict positive sentiment):")
print(coef_df.head(10)[['feature', 'coefficient']])

print("\nTop 10 Negative Features (predict negative sentiment):")
print(coef_df.tail(10)[['feature', 'coefficient']])


Coefficient analysis saved to ..\outputs\figures\lr_coefficients.png

Top 10 Positive Features (predict positive sentiment):
              feature  coefficient
10              great     0.020834
29   highly recommend     0.017613
30      review_length     0.015979
9       great product     0.015728
20          satisfied     0.014653
16              cheap     0.013912
11            quality     0.011269
26  exactly described     0.007873
21               love     0.007604
31        review_year     0.000206

Top 10 Negative Features (predict negative sentiment):
              feature  coefficient
1             product    -0.026361
3            terrible    -0.026969
13              waste    -0.027966
18              broke    -0.033307
24               work    -0.035711
17         doesn work    -0.036317
2        disappointed    -0.053186
4               worth    -0.055002
0   recommend product    -0.060264
15           material    -0.068212


### 6.2 XGBoost Feature Importance


In [10]:
# Analyze XGBoost feature importance
importance_df = analyze_xgboost_importance(
    xgb_model, feature_names, top_n=20,
    save_path=output_dir / "figures" / "xgb_importance.png"
)

print("\nTop 20 Most Important Features:")
print(importance_df)


Feature importance plot saved to ..\outputs\figures\xgb_importance.png

Top 20 Most Important Features:
              feature  importance
0   recommend product    0.119149
22     cheap material    0.042896
2        disappointed    0.038807
19            quickly    0.037831
18              broke    0.032872
1             product    0.032697
4               worth    0.032263
24               work    0.031604
17         doesn work    0.031295
3            terrible    0.029913
10              great    0.028554
13              waste    0.028271
29   highly recommend    0.027333
7        poor quality    0.026768
14              money    0.026649
23              doesn    0.026100
21               love    0.025467
6            returned    0.025358
26  exactly described    0.025298
20          satisfied    0.025005


## Step 7: Error Analysis


In [11]:
# Load original text for error analysis
data_path = Path("../data/processed/amazon_reviews_processed.parquet")
df_processed = pd.read_parquet(data_path)

# Create binary labels to match validation set indices
from modeling import create_binary_labels
df_labeled = create_binary_labels(df_processed)

# Get validation set indices
_, val_indices = train_test_split(
    np.arange(len(df_labeled)), test_size=0.2, random_state=42, stratify=df_labeled['sentiment']
)

# Get validation texts
val_texts = df_labeled.iloc[val_indices]['review_text_clean'].values

# Error analysis for Logistic Regression
print("=" * 80)
print("LOGISTIC REGRESSION ERROR ANALYSIS")
print("=" * 80)
errors_lr = error_analysis(y_val, y_pred_lr, val_texts, top_n=10)

# Error analysis for XGBoost
print("\n" + "=" * 80)
print("XGBOOST ERROR ANALYSIS")
print("=" * 80)
errors_xgb = error_analysis(y_val, y_pred_xgb, val_texts, top_n=10)


Label distribution:
sentiment
1    32392
0    10003
Name: count, dtype: int64
Positive: 32392, Negative: 10003
LOGISTIC REGRESSION ERROR ANALYSIS

Error Analysis:
  False Positives: 2523
  False Negatives: 1112

Top 10 False Positive Examples (Predicted Positive, Actually Negative):
  1. excellent quality exactly as described top quality i would definitely buy this again...
  2. fast shipping very satisfied fast shipping amazing value i would definitely buy this again...
  3. excellent quality best purchase highly recommend highly recommend i would definitely buy this again...
  4. love it works perfectly highly recommend top quality i would definitely buy this again...
  5. amazing value best purchase i would definitely buy this again...
  6. exactly as described great product amazing value love it i would definitely buy this again...
  7. excellent quality amazing value best purchase exactly as described i would definitely buy this again...
  8. fast shipping exactly as described i w

## Step 8: Save Models


In [12]:
# Save trained models
# Note: Make sure you've run cells 3, 7, and 9 first to define output_dir, lr_model, and xgb_model

# Define output_dir if not already defined (from Cell 3)
if 'output_dir' not in locals():
    output_dir = Path("../outputs")
    print("Note: output_dir was not defined. Using default: ../outputs")

models_dir = output_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Check if models exist before saving
if 'lr_model' not in locals():
    print("ERROR: lr_model is not defined. Please run Cell 7 first to train the Logistic Regression model.")
elif 'xgb_model' not in locals():
    print("ERROR: xgb_model is not defined. Please run Cell 9 first to train the XGBoost model.")
else:
    # Save models in binary format (.pkl files)
    print("Saving models in binary format (.pkl)...")
    save_model(lr_model, models_dir / "logistic_regression.pkl")
    save_model(xgb_model, models_dir / "xgboost.pkl")
    
    # Export model information in human-readable formats
    print("\nExporting model information in human-readable formats...")
    from modeling import export_model_info
    
    # Export Logistic Regression info
    export_model_info(
        lr_model, 
        "logistic_regression",
        metrics=metrics_lr if 'metrics_lr' in locals() else None,
        feature_names=feature_names if 'feature_names' in locals() else None,
        output_path=models_dir
    )
    
    # Export XGBoost info
    export_model_info(
        xgb_model,
        "xgboost",
        metrics=metrics_xgb if 'metrics_xgb' in locals() else None,
        feature_names=feature_names if 'feature_names' in locals() else None,
        output_path=models_dir
    )
    
    print("\n" + "="*80)
    print("Models saved successfully!")
    print("="*80)
    print("\nFiles created:")
    print(f"  Binary models (for loading/reuse):")
    print(f"    - {models_dir / 'logistic_regression.pkl'}")
    print(f"    - {models_dir / 'xgboost.pkl'}")
    print(f"\n  Human-readable information:")
    print(f"    - {models_dir / 'logistic_regression_info.json'}")
    print(f"    - {models_dir / 'logistic_regression_info.txt'}")
    print(f"    - {models_dir / 'xgboost_info.json'}")
    print(f"    - {models_dir / 'xgboost_info.txt'}")
    print("\nYou can open the .txt or .json files in any text editor to view model details!")


Saving models in binary format (.pkl)...
Model saved to ..\outputs\models\logistic_regression.pkl
Model saved to ..\outputs\models\xgboost.pkl

Exporting model information in human-readable formats...
Model information exported to:
  - JSON: ..\outputs\models\logistic_regression_info.json
  - Text: ..\outputs\models\logistic_regression_info.txt
Model information exported to:
  - JSON: ..\outputs\models\xgboost_info.json
  - Text: ..\outputs\models\xgboost_info.txt

Models saved successfully!

Files created:
  Binary models (for loading/reuse):
    - ..\outputs\models\logistic_regression.pkl
    - ..\outputs\models\xgboost.pkl

  Human-readable information:
    - ..\outputs\models\logistic_regression_info.json
    - ..\outputs\models\logistic_regression_info.txt
    - ..\outputs\models\xgboost_info.json
    - ..\outputs\models\xgboost_info.txt

You can open the .txt or .json files in any text editor to view model details!


## Step 9: Optional - SHAP Analysis

This section provides SHAP (SHapley Additive exPlanations) visualizations for model interpretability.


In [13]:
# Optional SHAP analysis
try:
    import shap
    
    print("Generating SHAP explanations...")
    print("Note: This may take a while for large datasets.")
    
    # Sample a subset for SHAP (SHAP can be slow on large datasets)
    n_shap_samples = min(100, X_val.shape[0])
    shap_indices = np.random.choice(X_val.shape[0], n_shap_samples, replace=False)
    X_val_shap = X_val[shap_indices]
    
    # SHAP for XGBoost (more interpretable with tree explainer)
    print("\nGenerating SHAP values for XGBoost...")
    explainer_xgb = shap.TreeExplainer(xgb_model)
    shap_values_xgb = explainer_xgb.shap_values(X_val_shap)
    
    # Summary plot
    shap.summary_plot(
        shap_values_xgb, X_val_shap, 
        feature_names=feature_names[:50],  # Limit to top 50 features for visualization
        max_display=20,
        show=False
    )
    plt.savefig(output_dir / "figures" / "shap_summary_xgb.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("SHAP summary plot saved to outputs/figures/shap_summary_xgb.png")
    
    # Bar plot of mean SHAP values
    shap.plots.bar(
        shap_values_xgb, 
        max_display=20,
        show=False
    )
    plt.savefig(output_dir / "figures" / "shap_bar_xgb.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("SHAP bar plot saved to outputs/figures/shap_bar_xgb.png")
    
except ImportError:
    print("SHAP not installed. Install with: pip install shap")
except Exception as e:
    print(f"SHAP analysis failed: {e}")
    print("This is an optional extension - continuing without SHAP...")


Generating SHAP explanations...
Note: This may take a while for large datasets.

Generating SHAP values for XGBoost...
SHAP summary plot saved to outputs/figures/shap_summary_xgb.png
SHAP analysis failed: The shap_values argument must be an Explanation object, Cohorts object, or dictionary of Explanation objects!
This is an optional extension - continuing without SHAP...


## Summary

### Model Performance Summary

Both models have been trained and evaluated:

1. **Logistic Regression**: Linear baseline with high interpretability
   - Provides coefficient analysis showing which features predict positive/negative sentiment
   - Good for understanding feature relationships

2. **XGBoost**: Nonlinear model capturing complex patterns
   - Generally performs better on complex datasets
   - Feature importance shows which features contribute most to predictions

### Key Insights

- Both models show good performance on sentiment classification
- Feature importance analysis reveals which words/phrases are most predictive
- Error analysis helps identify edge cases and model limitations
- SHAP analysis (optional) provides additional interpretability

### Next Steps

- Models are saved and can be used for inference
- All visualizations and metrics are saved in outputs/figures/
- Consider hyperparameter tuning for further improvements
